In [2]:
dem_path = "/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/data/raw/DEM.npz"
dem_21_path = "/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/data/raw/DEM21_opt.npz"


In [12]:
dem = np.load(dem_path)
dem_21 = np.load(dem_21_path)


print(dem.keys())
print(dem_21.keys())

KeysView(NpzFile '/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/data/raw/DEM.npz' with keys: dataset, validMask)
KeysView(NpzFile '/home/nikitachernysh/storage/Projects/lidar-archaeology-segmentation/data/raw/DEM21_opt.npz' with keys: dataset, validMask)


In [ ]:
import numpy as np
from scipy.signal import correlate2d

def find_offset_with_mask(sub_path, full_path):
    sub = np.load(sub_path)
    full = np.load(full_path)

    sub_dem = sub["dataset"].astype(np.float32)
    sub_mask = sub["validMask"].astype(bool)

    full_dem = full["dataset"].astype(np.float32)
    full_mask = full["validMask"].astype(bool)

    # Применяем маску: обнуляем невалидные пиксели
    sub_dem_masked = np.where(sub_mask, sub_dem, np.nan)
    full_dem_masked = np.where(full_mask, full_dem, np.nan)

    # Заменяем NaN на медиану по валидным пикселям
    def clean(a):
        valid = np.isfinite(a)
        med = np.nanmedian(a[valid])
        a[~valid] = med
        return a

    sub_dem = clean(sub_dem_masked)
    full_dem = clean(full_dem_masked)

    # Ограничиваем экстремальные значения
    def clip_percentile(a, pmin=1, pmax=99):
        lo, hi = np.nanpercentile(a, [pmin, pmax])
        return np.clip(a, lo, hi)

    sub_dem = clip_percentile(sub_dem)
    full_dem = clip_percentile(full_dem)

    # Нормализация по валидным пикселям
    def norm(a, mask):
        valid = mask.astype(bool)
        mean = np.mean(a[valid])
        std = np.std(a[valid]) + 1e-8
        a_norm = np.zeros_like(a, dtype=np.float32)
        a_norm[valid] = (a[valid] - mean) / std
        return a_norm

    sub_norm = norm(sub_dem, sub_mask)
    full_norm = norm(full_dem, full_mask)

    # Кросс-корреляция (учитываем только валидные области)
    corr = correlate2d(full_norm, sub_norm, mode="valid")
    y, x = np.unravel_index(np.argmax(corr), corr.shape)

    print(f"📍 Найденное смещение подмассива относительно полного DEM:")
    print(f"offset_y = {y}, offset_x = {x}")
    return y, x


In [ ]:
y, x = find_offset_with_mask(dem_path, dem_21_path)
print(f"offset_y = {y}, offset_x = {x}")
